In [1]:
import unicodedata
import numpy as np
import pandas as pd
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding
import os
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=30000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [3]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [4]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [5]:
len(data)

30000

In [6]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [7]:
en_sen="Im very happy."
preprossing(en_sen)

'<start> im very happy. <end>'

In [8]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start> درود بر تو. <end>'

In [9]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [10]:
vocab_size=2000
max_length=256
batch_size=20

token_en=Tokenizer(num_words=vocab_size,filters="")
token_fa=Tokenizer(num_words=vocab_size,filters="")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]

dataset = tf.data.Dataset.from_tensor_slices(((en_seq, decoder_inputs_array), decoder_targets_array))
dataset = dataset.shuffle(buffer_size=len(en_seq)).batch(batch_size).prefetch(tf.data.AUTOTUNE)



In [11]:
(x,dec_in),y=next(iter(dataset))
print(x.shape,dec_in.shape,y.shape)

(20, 256) (20, 255) (20, 255)


In [12]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True)(encoder_embedding)



decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm")
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])




In [13]:
decoder_dense=Dense(vocab_size,activation="softmax")
decoder_outputs=decoder_dense(decoder_outputs)

In [14]:
model=tf.keras.Model([encoder_inputs, ],decoder_outputs)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [15]:
history=model.fit([en_seq,decoder_inputs_array],decoder_targets_array,
                  batch_size=batch_size,epochs=25)

Epoch 1/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 532s 354ms/step - accuracy: 0.3606 - loss: 4.2106
Epoch 2/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 529s 353ms/step - accuracy: 0.4012 - loss: 3.7473
Epoch 3/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 589s 393ms/step - accuracy: 0.4255 - loss: 3.4347
Epoch 4/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 524s 349ms/step - accuracy: 0.4431 - loss: 3.1914
Epoch 5/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 499s 333ms/step - accuracy: 0.4595 - loss: 2.9846
Epoch 6/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 501s 334ms/step - accuracy: 0.4758 - loss: 2.7938
Epoch 7/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 505s 337ms/step - accuracy: 0.4924 - loss: 2.6112
Epoch 8/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 504s 336ms/step - accuracy: 0.5117 - loss: 2.4373
Epoch 9/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 503s 335ms/step - accuracy: 0.5338 - loss: 2.2691
Epoch 10/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 502s 335ms/step - accuracy: 0.5567 - loss: 2.1070
Epoch 11/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 501s 334ms/step - accuracy: 0.5825 -

In [19]:
reverse_fa = {v: k for k, v in token_fa.word_index.items()}

In [21]:
import pickle
model.save('Translator.keras')
pickle.dump(token_fa,open("token_fa.pkl","wb"))
pickle.dump(token_en,open("token_en.pkl","wb"))
pickle.dump(reverse_fa,open("reverse_fa.pkl","wb"))

In [23]:
from tensorflow.keras.models import load_model
model=load_model("Translator.keras")

In [24]:
import pickle

token_fa = pickle.load(open("token_fa.pkl","rb"))
token_en = pickle.load(open("token_en.pkl","rb"))
reverse_fa = pickle.load(open("reverse_fa.pkl","rb"))


In [25]:
encoder_model=tf.keras.Model(encoder_inputs,[state_h,state_c])

In [26]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_emb2 = decoder_embedding(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = tf.keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2, state_h2, state_c2]
)


In [27]:
def translate(sentence):

    sentence = preprossing(sentence)

    seq = token_en.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_length, padding="post")

    states = encoder_model.predict(seq)

    target_seq = np.array([[token_fa.word_index["<start>"]]])

    stop = False
    decoded = ""

    while not stop:

        output_tokens, h, c = decoder_model.predict([target_seq] + states)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_fa.get(sampled_token_index, "")

        if sampled_word == "<end>" or len(decoded.split()) > max_length:
            stop = True
        else:
            decoded += " " + sampled_word

        target_seq = np.array([[sampled_token_index]])
        states = [h, c]

    return decoded


In [28]:
print(translate("I love you"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
 من تو رو دوست دارم


In [38]:
print(translate('you can'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
 تو مي توني


In [ ]:
import sacrebleu